# Decision Trees — From Intuition to Implementation

**Duration:** ~2 hours
**Libraries:** NumPy · Pandas · Matplotlib · Seaborn · Scikit-learn

---

### What are we doing today?

Yesterday you predicted a **number** (exam score) using Linear Regression.

Today you'll predict a **category** — Pass or Fail, Yes or No, Spam or Not Spam.
This is called **Classification**, and Decision Trees are one of the most
intuitive ways to do it.

> *A Decision Tree asks a series of yes/no questions about your data
> and arrives at an answer — exactly like how a human makes decisions.*

---

### By the end of this notebook you will:
- Understand how a Decision Tree makes decisions (with zero math anxiety)
- Know what Gini Impurity and Information Gain mean in plain English
- Build, visualise, and evaluate a Decision Tree classifier
- Understand overfitting and how to fix it with pruning
- Complete a mini project on a second real-world dataset


## Part 1 — The Intuition (No Code Yet!)

### 1.1 You Already Know Decision Trees

Every time you make a decision like this, you're running a decision tree:

```
         Is it raining?
         /           \
       YES             NO
        |               |
  Take umbrella?   Is it hot?
   /        \       /      \
 YES         NO   YES       NO
  |           |    |         |
Take it    Skip  Wear        Wear
           it    shorts    a jacket
```

A Decision Tree in ML does the **exact same thing** — but it learns
the questions and thresholds automatically from your data.

---

### 1.2 Key Vocabulary

| Term | Plain English |
|------|--------------|
| **Root Node** | The very first question (top of the tree) |
| **Branch** | A path taken based on the answer to a question |
| **Internal Node** | Any question that's not the last one |
| **Leaf Node** | The final answer — a class label |
| **Depth** | How many questions deep the tree goes |
| **Splitting** | Choosing which question to ask at each node |

---

### 1.3 How Does the Tree Choose Which Question to Ask?

It picks the question that creates the **purest** groups — i.e., after
splitting, each group should contain mostly one class.

We measure "impurity" using **Gini Impurity**:
- Gini = 0 → perfectly pure (all one class) ✅
- Gini = 0.5 → maximally impure (50/50 split) ❌

The tree always picks the split that **lowers Gini the most**.


## Part 2 — Dataset 1: Student Pass/Fail Prediction

### 2.1 Load Libraries and Create Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (9, 5)
np.random.seed(42)

print("✅ Libraries imported!")

In [ ]:
# ── CREATE DATASET ──────────────────────────────────────────────
# 200 students with 4 features
n = 200

study_hours   = np.round(np.random.uniform(1, 10, n), 1)
sleep_hours   = np.round(np.random.uniform(4, 9,  n), 1)
attendance    = np.random.randint(50, 101, n)          # % attendance
prev_score    = np.round(np.random.uniform(40, 100, n), 1)

# Pass rule: study>=5 AND (attendance>=75 OR prev_score>=65)
passed = (
    (study_hours >= 5) &
    ((attendance >= 75) | (prev_score >= 65))
).astype(int)

# Add a little noise so it's not perfectly separable
flip_idx = np.random.choice(n, size=15, replace=False)
passed[flip_idx] = 1 - passed[flip_idx]

df = pd.DataFrame({
    "study_hours" : study_hours,
    "sleep_hours" : sleep_hours,
    "attendance"  : attendance,
    "prev_score"  : prev_score,
    "result"      : passed
})
df["result_label"] = df["result"].map({1: "Pass", 0: "Fail"})

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["result_label"].value_counts())
print(f"\nFirst 8 rows:")
df.head(8)

### 2.2 Explore the Data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
features = ["study_hours", "sleep_hours", "attendance", "prev_score"]
colors   = {"Pass": "#3ecfb2", "Fail": "#e06060"}

for ax, feat in zip(axes.flatten(), features):
    for label, grp in df.groupby("result_label"):
        ax.hist(grp[feat], bins=15, alpha=0.6,
                label=label, color=colors[label], edgecolor="white")
    ax.set_title(f"{feat} by Result", fontweight="bold")
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Feature Distributions — Pass vs Fail", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 5))
corr = df[features + ["result"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Which features correlate most with 'result'?")
print(corr["result"].drop("result").sort_values(ascending=False))

### 2.3 Prepare Data — Train / Test Split

In [ ]:
X = df[features]
y = df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")
print()
print("Class balance in training set:")
print(pd.Series(y_train).map({1:"Pass",0:"Fail"}).value_counts())

### 2.4 Train the Decision Tree

In [ ]:
# max_depth=3 keeps the tree readable — we'll experiment with depth later
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

print("✅ Decision Tree trained!")
print(f"   Tree depth    : {dt.get_depth()}")
print(f"   Leaf nodes    : {dt.get_n_leaves()}")
print(f"   Features used : {list(X.columns)}")

### 2.5 Visualise the Tree

This is the best part — you can **read exactly what the model learned**.


In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    dt,
    feature_names=features,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=True,
    proportion=False,
)
plt.title("Decision Tree — Student Pass/Fail", fontsize=15, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

print("Blue nodes → leaning towards Pass")
print("Orange nodes → leaning towards Fail")
print("Darker colour = purer node (more confident prediction)")

In [ ]:
# Text version — great for reading the exact rules
print("=== DECISION RULES (text format) ===\n")
rules = export_text(dt, feature_names=features)
print(rules)

### 2.6 Make Predictions & Evaluate

In [ ]:
y_pred = dt.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"Accuracy on test set: {acc*100:.1f}%")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))

In [ ]:
# Confusion Matrix — the most informative evaluation plot
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly predicted Pass) : {tp}")
print(f"True Negatives  (correctly predicted Fail) : {tn}")
print(f"False Positives (predicted Pass, actually Fail) : {fp}  ← Type I error")
print(f"False Negatives (predicted Fail, actually Pass) : {fn}  ← Type II error")

In [ ]:
# Feature importance — which feature mattered most?
importance = pd.Series(dt.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importance.plot(kind="barh", color=["#e06060","#f0a040","#3ecfb2","#7c6af7"])
plt.title("Feature Importance", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.axvline(0, color="white", linewidth=0.5)
plt.tight_layout()
plt.show()

print("Higher = the tree relied on this feature more for its decisions.")

## Part 3 — Overfitting: The #1 Problem with Decision Trees

### What is Overfitting?

A deep tree memorises the **training data perfectly** — including its noise —
but fails on new data. This is called **overfitting**.

```
  UNDERFIT              GOOD FIT              OVERFIT
  (too simple)          (just right)          (too complex)

  High train error      Low train error       Very low train error
  High test  error      Low test  error       High test  error
  
  The tree is           The tree learned      The tree memorised
  too shallow           the real pattern      the training noise
```

Let's see this happen live by trying different tree depths.


In [ ]:
depths     = range(1, 16)
train_accs = []
test_accs  = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, "o-", color="#7c6af7", linewidth=2, label="Train Accuracy")
plt.plot(depths, test_accs,  "s-", color="#3ecfb2", linewidth=2, label="Test Accuracy")
plt.axvline(x=3, color="#f0a040", linestyle="--", linewidth=1.5, label="Our model (depth=3)")
plt.fill_between(depths, train_accs, test_accs, alpha=0.08, color="#e06060")
plt.title("Accuracy vs Tree Depth — Spotting Overfitting", fontsize=13, fontweight="bold")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.xticks(depths)
plt.legend()
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(test_accs)]
print(f"Best test accuracy at depth = {best_depth}")
print("Notice: after a certain depth, train accuracy keeps rising but test accuracy drops!")
print("That gap = overfitting.")

## Part 4 — Dataset 2: Iris Flower Classification

Now let's apply everything to a **famous real-world dataset** — the Iris dataset.

**Task:** Classify flowers into 3 species based on petal and sepal measurements.
- *Iris Setosa*
- *Iris Versicolor*
- *Iris Virginica*

This is a **multi-class classification** problem (3 classes instead of 2).


In [ ]:
from sklearn.datasets import load_iris

# Load the built-in Iris dataset
iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = iris.target
iris_df["species_name"] = iris_df["species"].map(
    {0: "Setosa", 1: "Versicolor", 2: "Virginica"}
)

print(f"Shape: {iris_df.shape}")
print(f"\nClass distribution:")
print(iris_df["species_name"].value_counts())
print()
iris_df.head(8)

In [ ]:
# Pairplot — visualise all feature combinations at once
sns.pairplot(iris_df, hue="species_name",
             vars=iris.feature_names,
             palette={"Setosa":"#7c6af7","Versicolor":"#3ecfb2","Virginica":"#f0a040"},
             plot_kws={"alpha":0.7, "s":40})
plt.suptitle("Iris Dataset — Pairplot", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Which pair of features separates the species best?")

In [ ]:
# Train / test split
X_iris = iris_df[iris.feature_names]
y_iris = iris_df["species"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.25, random_state=42, stratify=y_iris
)

# Train tree
iris_dt = DecisionTreeClassifier(max_depth=4, random_state=42)
iris_dt.fit(X_tr, y_tr)

y_iris_pred = iris_dt.predict(X_te)
print(f"Iris Accuracy: {accuracy_score(y_te, y_iris_pred)*100:.1f}%")
print()
print(classification_report(y_te, y_iris_pred,
      target_names=["Setosa","Versicolor","Virginica"]))

In [ ]:
# Visualise the Iris tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    iris_dt,
    feature_names=iris.feature_names,
    class_names=["Setosa","Versicolor","Virginica"],
    filled=True, rounded=True, fontsize=9, ax=ax
)
plt.title("Decision Tree — Iris Species Classification",
          fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for 3-class problem
fig, ax = plt.subplots(figsize=(6, 5))
cm_iris = confusion_matrix(y_te, y_iris_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_iris,
                               display_labels=["Setosa","Versicolor","Virginica"])
disp.plot(ax=ax, colorbar=False, cmap="Purples")
ax.set_title("Iris Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Part 5 — Key Concepts Summary

```
┌─────────────────────────────────────────────────────────────────┐
│               DECISION TREE — CONCEPT MAP                      │
├──────────────────────┬──────────────────────────────────────────┤
│  How it learns       │  Finds questions (splits) that create    │
│                      │  the purest groups (lowest Gini)         │
├──────────────────────┼──────────────────────────────────────────┤
│  Strengths           │  ✅ Interpretable — you can read the     │
│                      │     rules directly                       │
│                      │  ✅ No feature scaling needed            │
│                      │  ✅ Handles both numbers and categories  │
├──────────────────────┼──────────────────────────────────────────┤
│  Weaknesses          │  ❌ Prone to overfitting (deep trees)    │
│                      │  ❌ Small data changes = very            │
│                      │     different tree                       │
├──────────────────────┼──────────────────────────────────────────┤
│  Fix for overfitting │  max_depth, min_samples_split,           │
│                      │  min_samples_leaf  (pruning)             │
├──────────────────────┼──────────────────────────────────────────┤
│  What comes next     │  Random Forest = many trees voting       │
│                      │  together → fixes overfitting            │
└──────────────────────┴──────────────────────────────────────────┘
```


## 🧪 Challenges

### Challenge 1 — Tune the Tree (Easy)
Train the student dataset tree with `max_depth=5` and `max_depth=10`.
Compare test accuracy for all three depths (3, 5, 10) in a single print statement.
Which depth gives the best test accuracy?


In [ ]:
# Challenge 1: Try different depths
for depth in [3, 5, 10]:
    # your code here
    pass

### Challenge 2 — Predict a New Student (Easy)
A new student has:
- `study_hours = 6.0`
- `sleep_hours = 7.5`
- `attendance = 80`
- `prev_score = 72`

Will they Pass or Fail? Use the trained `dt` model to predict.
Also print the **probability** of passing using `predict_proba()`.


In [ ]:
# Challenge 2: Predict for a new student
new_student = pd.DataFrame({
    "study_hours" : [6.0],
    "sleep_hours" : [7.5],
    "attendance"  : [80],
    "prev_score"  : [72]
})

# prediction = dt.predict(new_student)
# probability = dt.predict_proba(new_student)
# Your code here

### Challenge 3 — Add a New Feature (Medium)
Add a column `assignments_done` (integer, random between 0 and 10) to the
student dataset. Retrain the tree. Does the new feature appear in the
feature importance chart? Does accuracy improve?


In [ ]:
# Challenge 3: New feature — assignments_done
np.random.seed(7)
df["assignments_done"] = np.random.randint(0, 11, len(df))

# Update features list, retrain, evaluate, plot importance
# Your code here

### Challenge 4 — Decision Boundary Plot (Hard)
Using only `study_hours` and `prev_score` as features, train a tree and
plot the **decision boundary** — the regions of the feature space
where the model predicts Pass vs Fail.

*Hint: create a meshgrid using `np.meshgrid`, predict on every point, and
use `plt.contourf()` to colour the regions.*


In [ ]:
# Challenge 4: Decision boundary visualisation
X2 = df[["study_hours", "prev_score"]]
y2 = df["result"]

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)

dt2 = DecisionTreeClassifier(max_depth=4, random_state=42)
dt2.fit(X2_tr, y2_tr)

# Create meshgrid
h = 0.1
x_min, x_max = X2["study_hours"].min()-0.5, X2["study_hours"].max()+0.5
y_min, y_max = X2["prev_score"].min()-2,    X2["prev_score"].max()+2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))

# Predict on every grid point
Z = dt2.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
scatter = plt.scatter(X2["study_hours"], X2["prev_score"],
                       c=y2, cmap="coolwarm", edgecolor="white", s=60)
plt.xlabel("Study Hours")
plt.ylabel("Previous Score")
plt.title("Decision Boundary — Study Hours vs Previous Score",
          fontsize=13, fontweight="bold")
fail_patch = mpatches.Patch(color="#d45f5f", alpha=0.5, label="Fail region")
pass_patch = mpatches.Patch(color="#5f9fd4", alpha=0.5, label="Pass region")
plt.legend(handles=[fail_patch, pass_patch])
plt.tight_layout()
plt.show()

## Wrap-Up

Today you learned:

- ✅ How Decision Trees make decisions — splitting by Gini impurity
- ✅ Key vocabulary: root, branch, internal node, leaf, depth
- ✅ How to train, visualise, and **read** a Decision Tree
- ✅ Confusion matrix and classification report
- ✅ Feature importance — which inputs matter most
- ✅ Overfitting — what it looks like and how to fix it with pruning
- ✅ Multi-class classification on the Iris dataset
- ✅ Decision boundary visualisation

### What's coming next
**Random Forests** — the natural upgrade to Decision Trees.
Instead of one tree, we grow hundreds of trees and let them vote.
It fixes overfitting and is one of the most powerful ML algorithms in practice.
